# Chapter 15: Instrumental Variables Estimation and Two Stage Least Squares

Topics: IV in simple models, additional exogenous regressors, 2SLS, exogeneity testing, overidentification.

In [ ]:
import numpy as np
import pandas as pd
import wooldridge as woo
import matplotlib.pyplot as plt
import os
from linearmodels.iv import IV2SLS, IVLIML

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 15.1 IV in Simple Regression

In [ ]:
df = woo.dataWoo('wage2').dropna()

# OLS (biased: educ endogenous)
ols = IV2SLS.from_formula('lwage ~ 1 + exper + [educ ~ educ]', data=df).fit()

# IV: instrument educ with sibs (siblings)
iv = IV2SLS.from_formula('lwage ~ 1 + exper + [educ ~ sibs]', data=df).fit()

print("OLS:")
print(ols.summary)
print("\nIV (instrument: sibs):")
print(iv.summary)

## 15.3 Two Stage Least Squares

In [ ]:
df = woo.dataWoo('mroz').dropna()

# 2SLS: instrument educ with fatheduc and motheduc
iv2sls = IV2SLS.from_formula(
    'lwage ~ 1 + exper + expersq + [educ ~ fatheduc + motheduc]',
    data=df
).fit()
print("2SLS Results:")
print(iv2sls.summary)

## First Stage: Instrument Strength

In [ ]:
import statsmodels.formula.api as smf

first_stage = smf.ols('educ ~ exper + expersq + fatheduc + motheduc', data=df).fit()
print(f"First Stage F-stat on instruments: {first_stage.fvalue:.2f}")
print(f"(Rule of thumb: F > 10 for strong instruments)")
print(first_stage.summary())

## 15.5 Testing Overidentifying Restrictions (Sargan Test)

In [ ]:
# Sargan-Hansen J-test: instruments valid?
print(iv2sls.wooldridge_overid)